# 2.5 — Random Forest + export (Forma 2, features do raw)

**Mesmo modelo da Forma 1** (Random Forest + StandardScaler + export **micromlgen**), agora
sobre as features que nós mesmos extraímos do raw no notebook **2.4** (`features_from_raw.csv`,
12 colunas de feature + `rodada` + `split`).

Ordem deste notebook: (1) um **baseline** simples, para saber se o problema é trivial antes de
comemorar qualquer acurácia; (2) comparação entre **três conjuntos de features**, que é o
exercício central — revela por que `mean_*`/`rms_mag` enganam aqui; (3) métricas, matriz de
confusão e importância; (4) validação mais rigorosa com `LeaveOneGroupOut` por rodada e a curva
de aprendizado; (5) export para o ESP32, já com o conjunto de features corrigido.

> **Atende Sprint 4:** item 2 (treino/teste + métricas + matriz de confusão) e item 3
> (feature importance + interpretação). A comparação com a Forma 1 (notebook 1.5) é o ponto
> didático: **mesmo problema, mesmo modelo, caminhos de dados diferentes**.

In [ ]:
!pip install -q pandas scikit-learn matplotlib micromlgen

## 1) Carregar as features (saída do 2.4)

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

try:
    from google.colab import files
    enviados = files.upload()
    nome = list(enviados.keys())[0]
except Exception:
    nome = "features_from_raw.csv"

df = pd.read_csv(nome)
# "rodada" e string no CSV ("01", "02"...), mas o read_csv infere int e perde o
# zero a esquerda. Forcamos de volta para bater com o nome do arquivo original.
df["rodada"] = df["rodada"].astype(str).str.zfill(2)
print(df.groupby(["label", "rodada", "split"]).size())
df.head()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance

## 2) Baseline: o problema é trivial?

Antes de qualquer Random Forest, um limiar bobo em cima de UMA feature. Se ele já resolve, a
conclusão honesta é "o problema é trivial" — não "o modelo é ótimo". Parado na mesa vs.
chacoalhado difere em `std_mag` por 2–3 ordens de grandeza; é razoável esperar que baste um
único limiar.

In [ ]:
CLASSES = {"normal": 0, "anomalo": 1}
df = df[df["label"].isin(CLASSES)].copy()
df["y"] = df["label"].map(CLASSES)

tr = df[df["split"] == "treino"]
te = df[df["split"] == "teste"]
print(f"Treino: {len(tr)} | Teste: {len(te)}")

piso = DummyClassifier(strategy="most_frequent").fit(tr[["std_mag"]], tr["y"])
acc_piso = accuracy_score(te["y"], piso.predict(te[["std_mag"]]))
print(f"Piso (classe majoritaria):              acuracia={acc_piso:.3f}")

arvore1 = DecisionTreeClassifier(max_depth=1, random_state=42).fit(tr[["std_mag"]], tr["y"])
acc_arvore1 = accuracy_score(te["y"], arvore1.predict(te[["std_mag"]]))
limiar = arvore1.tree_.threshold[0]
print(f"Arvore de profundidade 1 (so std_mag):  acuracia={acc_arvore1:.3f}")
print(f"  limiar aprendido: std_mag <= {limiar:.4f} -> normal | > {limiar:.4f} -> anomalo")
print()
print("Se o Random Forest abaixo nao superar isso por uma margem clara, o problema binario")
print("normal x anomalo (motor parado x chacoalhado) e trivial — o interessante vem so com")
print("as 5 classes do motor (notebook 2.6).")

## 3) Três conjuntos de features (o exercício deste notebook)

O `2.4` gerou 12 features e não removeu nada: as 7 do edge (`mean_ax/ay/az`, `std_ax/ay/az`,
`rms_mag`) mais 5 novas sobre a magnitude do sinal. Treinamos o MESMO Random Forest com três
conjuntos e comparamos:

| conjunto | composição | o que queremos ver |
|---|---|---|
| `FEATURES_ORIG` | as 7 do edge | acurácia alta — mas por qual motivo? |
| `FEATURES_AC` | sem `mean_*`; `std_mag` no lugar de `rms_mag`; + as outras 4 novas | acurácia deve se manter |
| `FEATURES_TUDO` | as 12 | a importância (seção 5) revela o que está acontecendo |

Duas contas para ter em mente ao ler o resultado:

- **`mean_ax/ay/az` codificam orientação, não vibração.** Parado, `mean_az ≈ 9,81`. Um modelo
  que dependa disso quebra assim que o sensor for montado em outro ângulo.
- **`rms_mag` satura na gravidade:** `rms_mag = sqrt(mean(ax²+ay²+az²)) ≈ sqrt(9,81² + σ²)`.
  Para σ = 1 m/s² isso vai de 9,81 para 9,861 — variação de **0,5%**. A mesma informação em
  `std_mag` (a componente AC, sem a gravidade) vai de ~0 para 1,0.

In [ ]:
FEATURES_ORIG = ["mean_ax", "mean_ay", "mean_az", "std_ax", "std_ay", "std_az", "rms_mag"]
FEATURES_AC   = ["std_ax", "std_ay", "std_az", "std_mag", "p2p_mag", "crest_mag", "kurt_mag", "zcr_mag"]
FEATURES_TUDO = FEATURES_ORIG + [f for f in FEATURES_AC if f not in FEATURES_ORIG]

def treina_avalia(features):
    scaler = StandardScaler().fit(tr[features].values)
    X_tr = scaler.transform(tr[features].values)
    X_te = scaler.transform(te[features].values)
    clf = RandomForestClassifier(n_estimators=20, max_depth=8, random_state=42)
    clf.fit(X_tr, tr["y"].values)
    y_pred = clf.predict(X_te)
    return clf, scaler, y_pred

resultados = {}
for nome, feats in [("FEATURES_ORIG", FEATURES_ORIG), ("FEATURES_AC", FEATURES_AC), ("FEATURES_TUDO", FEATURES_TUDO)]:
    clf, scaler, y_pred = treina_avalia(feats)
    acc = accuracy_score(te["y"], y_pred)
    f1m = f1_score(te["y"], y_pred, average="macro")
    resultados[nome] = {"clf": clf, "scaler": scaler, "features": feats, "y_pred": y_pred,
                        "acuracia": acc, "f1_macro": f1m}
    print(f"{nome:15s} ({len(feats):2d} features): acuracia={acc:.3f}  f1_macro={f1m:.3f}")

## 4) Métricas e matriz de confusão (conjunto `FEATURES_TUDO`)

Olhamos em detalhe o conjunto com as 12 features — é o que alimenta a importância da próxima
seção, que revela qual grupo de features o modelo realmente usou para decidir.

In [ ]:
y_pred_tudo = resultados["FEATURES_TUDO"]["y_pred"]
print(classification_report(te["y"], y_pred_tudo, target_names=["normal", "anomalo"]))
ConfusionMatrixDisplay(confusion_matrix(te["y"], y_pred_tudo),
                       display_labels=["normal", "anomalo"]).plot()
plt.show()

## 5) Feature importance — qual grupo o modelo realmente usou?

In [ ]:
clf_tudo = resultados["FEATURES_TUDO"]["clf"]
scaler_tudo = resultados["FEATURES_TUDO"]["scaler"]
X_test_tudo_s = scaler_tudo.transform(te[FEATURES_TUDO].values)

imp_nativa = pd.Series(clf_tudo.feature_importances_, index=FEATURES_TUDO).sort_values()
perm = permutation_importance(clf_tudo, X_test_tudo_s, te["y"].values, n_repeats=20, random_state=42)
imp_perm = pd.Series(perm.importances_mean, index=FEATURES_TUDO).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
imp_nativa.plot.barh(ax=axes[0], color="tab:blue");  axes[0].set_title("Importância nativa (RF)")
imp_perm.plot.barh(ax=axes[1], color="tab:orange");  axes[1].set_title("Permutation importance (teste)")
plt.tight_layout(); plt.show()

**Interpretação:** se `mean_az` (ou outro `mean_*`) aparecer perto do topo, é sinal de que o
Random Forest achou um atalho — está classificando *postura*, não vibração; quebraria assim que
o sensor fosse montado em outro ângulo. O fisicamente esperado é `std_mag`, `kurt_mag` e
`std_*` no topo: vibração eleva a intensidade e a variabilidade efetivas, enquanto a média quase
não muda (Aula 14, slides 12–13).

Guarde essa conclusão: no notebook **2.6** (5 classes do motor, `app17-11`), o mesmo `mean_*`
vira a família de features que resolve as classes de inclinação — a mesma feature, dois
veredictos opostos, dependendo da pergunta que o modelo precisa responder.

**Dali em diante, este notebook segue só com `FEATURES_AC`** — o conjunto fisicamente honesto,
sem o atalho de orientação — para a validação mais rigorosa e para o que vai para o ESP32.

## 6) Validação robusta: `LeaveOneGroupOut` por rodada

O split do 2.4 já é por rodada (holdout da última). Aqui vamos além: deixamos CADA rodada de
fora, uma de cada vez, e olhamos a variação da acurácia entre elas — revela se o modelo
generaliza entre sessões de coleta ou decorou uma rodada específica. Precisa de pelo menos 2
rodadas por classe; com menos, degenera para 1 fold só.

`rodada` é numerada dentro de cada classe (a "rodada 01" de `normal` e a "rodada 01" de
`anomalo` vêm de sessões diferentes — o coletor grava uma classe por execução). Por isso um
fold pode conter as duas classes com o mesmo número: não é vazamento, só um agrupamento um
pouco mais conservador que o ideal.

In [ ]:
X_ac = df[FEATURES_AC].values
y_ac = df["y"].values
grupos = df["rodada"].astype(str)

n_rodadas_por_classe = df.groupby("label")["rodada"].nunique()
print("Rodadas por classe:")
print(n_rodadas_por_classe)
if (n_rodadas_por_classe < 2).any():
    print("\nAVISO: alguma classe tem menos de 2 rodadas — o LeaveOneGroupOut abaixo tem poucos")
    print("folds e nao e conclusivo. Colete mais rodadas para um resultado confiavel.\n")

logo = LeaveOneGroupOut()
accs, f1s = [], []
for fold, (idx_tr, idx_te) in enumerate(logo.split(X_ac, y_ac, groups=grupos)):
    scaler_f = StandardScaler().fit(X_ac[idx_tr])
    clf_f = RandomForestClassifier(n_estimators=20, max_depth=8, random_state=42)
    clf_f.fit(scaler_f.transform(X_ac[idx_tr]), y_ac[idx_tr])
    y_pred_f = clf_f.predict(scaler_f.transform(X_ac[idx_te]))

    acc_f = accuracy_score(y_ac[idx_te], y_pred_f)
    f1_f  = f1_score(y_ac[idx_te], y_pred_f, average="macro", zero_division=0)
    rodada_fora  = grupos.iloc[idx_te[0]]
    classes_fora = sorted(df["label"].iloc[idx_te].unique())
    accs.append(acc_f); f1s.append(f1_f)
    print(f"fold {fold+1:2d} (rodada {rodada_fora} fora, classes {classes_fora}): "
          f"acuracia={acc_f:.3f}  f1_macro={f1_f:.3f}")

print(f"\nMedia entre folds: acuracia={np.mean(accs):.3f} (+-{np.std(accs):.3f})  "
     f"f1_macro={np.mean(f1s):.3f}")

## 7) Curva de aprendizado: quantas rodadas são suficientes?

Treinamos com 1, 2, 3… rodadas de treino por classe e avaliamos sempre no MESMO holdout (a
última rodada, coluna `split`). Quando a curva achatar, coletar mais rodadas deixa de ajudar —
é a resposta baseada em medição para "de quantos dados eu preciso", em vez de regra de bolso.

In [ ]:
rodadas_normal  = sorted(tr[tr["label"] == "normal"]["rodada"].unique())
rodadas_anomalo = sorted(tr[tr["label"] == "anomalo"]["rodada"].unique())
n_max = min(len(rodadas_normal), len(rodadas_anomalo))

if n_max < 2:
    print(f"So {n_max} rodada(s) de treino disponivel(is) por classe — colete mais para ver a curva.")
else:
    accs_curva, f1s_curva = [], []
    for k in range(1, n_max + 1):
        usa_rodadas = set(rodadas_normal[:k]) | set(rodadas_anomalo[:k])
        tr_k = tr[tr["rodada"].isin(usa_rodadas)]

        scaler_k = StandardScaler().fit(tr_k[FEATURES_AC].values)
        clf_k = RandomForestClassifier(n_estimators=20, max_depth=8, random_state=42)
        clf_k.fit(scaler_k.transform(tr_k[FEATURES_AC].values), tr_k["y"].values)

        y_pred_k = clf_k.predict(scaler_k.transform(te[FEATURES_AC].values))
        acc_k = accuracy_score(te["y"], y_pred_k)
        f1_k  = f1_score(te["y"], y_pred_k, average="macro")
        accs_curva.append(acc_k); f1s_curva.append(f1_k)
        print(f"{k} rodada(s)/classe ({len(tr_k)} janelas): acuracia={acc_k:.3f}  f1_macro={f1_k:.3f}")

    plt.figure(figsize=(7, 4))
    plt.plot(range(1, n_max + 1), accs_curva, marker="o", label="acuracia")
    plt.plot(range(1, n_max + 1), f1s_curva, marker="s", label="f1_macro")
    plt.xlabel("rodadas de treino por classe"); plt.ylabel("metrica no holdout")
    plt.xticks(range(1, n_max + 1)); plt.ylim(0, 1.05); plt.legend()
    plt.title("Curva de aprendizado por numero de rodadas")
    plt.tight_layout(); plt.show()

## 8) Exportar para o ESP32 (micromlgen)

Exportamos o modelo treinado com `FEATURES_AC` (seção 3) — o conjunto sem o atalho de
orientação, o que faz sentido embarcar de verdade. Gera os mesmos dois headers da Forma 1.

In [ ]:
from micromlgen import port

clf_final    = resultados["FEATURES_AC"]["clf"]
scaler_final = resultados["FEATURES_AC"]["scaler"]

with open("AIoTVibracaoRF_micromlgen.hpp", "w") as f:
    f.write(port(clf_final))

def gerar_scaler_hpp(scaler, features):
    n = len(features)
    means  = ", ".join(f"{m:.10f}f" for m in scaler.mean_)
    scales = ", ".join(f"{s:.10f}f" for s in scaler.scale_)
    lista_features = ", ".join(features)
    return f'''#ifndef STANDARD_SCALER_HPP
#define STANDARD_SCALER_HPP
// StandardScaler de {n} features (FEATURES_AC): {lista_features}
namespace Scaler {{
    const static float means[{n}]  = {{ {means} }};
    const static float scales[{n}] = {{ {scales} }};
    inline void std(const float* input, float* output) {{
        for (int i = 0; i < {n}; i++) output[i] = (input[i] - means[i]) / scales[i];
    }}
}}
#endif
'''

with open("AIoTVibracaoScaler.hpp", "w") as f:
    f.write(gerar_scaler_hpp(scaler_final, FEATURES_AC))

try:
    from google.colab import files
    files.download("AIoTVibracaoRF_micromlgen.hpp")
    files.download("AIoTVibracaoScaler.hpp")
except Exception:
    print("Arquivos gerados na pasta atual (fora do Colab).")